# Scratchpad
A place to experiment with agent calls in a modular way before inserting them into the codebase.

In [1]:
from dotenv import load_dotenv
from IPython.display import Image, display
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import MemorySaver
from langchain_mcp_adapters.client import MultiServerMCPClient
import os

load_dotenv(override=True)


True

## Parsing Experiments

In [2]:
import asyncio
from quarterly_report_parse_result import QuarterlyReportParseResult
from llm_models import get_default_model
from pdf_reader import load_pdf_with_markdown_tables, merge_pages
from table_parser import (
    turn_tables_to_string,
    extract_tables_from_report,
)
from date_parser import (
    get_quarter_parsing_parameters,
    extract_quarter_info,
)

parser_system_prompt = """
You are a financial analyst.

You are given a 10Q document.

You are tasked with extracting financial statements from 10Q documents, so that your team can run fundamental analysis using that data.

Documents often have multiple values for each field, which corresponds to different dates. 
You should only extract the value that is for the most recent date.

Sometimes reports will use a dash or a line to represent zero. 
Just use 0. Do not try to infer the value.

Sometimes values will be surrounded by parentheses. This represents a negative value. 
Use the negative value of the number inside the parentheses.
"""
parser_agent = create_agent(
    system_prompt=parser_system_prompt,
    model=get_default_model(),
    response_format=QuarterlyReportParseResult,
)

# Runs the parser on one page of the document at a time
# At the end, it merges the results together
async def run_parser_page_by_page() -> QuarterlyReportParseResult:
    quarterly_report_pages = load_pdf_with_markdown_tables("sandbox/quarterly_report.pdf")
    requests = [run_parser_on_page_of_quarterly_report(page) for page in quarterly_report_pages]
    results = await asyncio.gather(*requests)

    merged_result = QuarterlyReportParseResult()
    for result in results:
        merged_result = merged_result.merge(result)
    return merged_result

async def run_parser_on_page_of_quarterly_report(page: str):
    message = f"""
Here is a page from a 10Q document that I would like you to parse for financial statements:

{page}
"""

    result = await parser_agent.ainvoke({"messages": [{"role": "user", "content": message}]})
    return result["structured_response"]

# Runs the parser on the whole document at once
def run_parser_on_whole_document() -> QuarterlyReportParseResult:
    message = f"""
Here is a 10Q document that I would like you to parse:

{merge_pages(load_pdf_with_markdown_tables("sandbox/quarterly_report.pdf"))}
"""

    result = parser_agent.invoke({"messages": [{"role": "user", "content": message}]})
    return result["structured_response"]

async def run_parser_on_all_tables_at_once() -> QuarterlyReportParseResult:
    pages = load_pdf_with_markdown_tables("sandbox/quarterly_report.pdf")
    quarter_info = await extract_quarter_info(pages)
    params = get_quarter_parsing_parameters(quarter_info)
    tables = await extract_tables_from_report(pages, params)

    message = f"""
Here is are all of the tables from a 10Q document. I would like you to parse them for financial statements:

{turn_tables_to_string(tables)}
"""

    result = await parser_agent.ainvoke({"messages": [{"role": "user", "content": message}]})
    return result["structured_response"]


In [3]:
# Run comparisons
from quarterly_report_parse_result import count_populated_fields, get_diffs
from document_parser import run_parser_table_by_table


# table_by_table_quarterly_report_result = await run_parser_table_by_table()
# all_tables_at_once_quarterly_report_result = await run_parser_on_all_tables_at_once()
# quarterly_report_result = run_parser_on_whole_document()

# print(f"Quarterly Report Result: {count_populated_fields(quarterly_report_result)}")
# print(f"Table by Table Quarterly Report Result: {count_populated_fields(table_by_table_quarterly_report_result)}")
# print(f"All Tables at Once Quarterly Report Result: {count_populated_fields(all_tables_at_once_quarterly_report_result)}")

# diffs = get_diffs(quarterly_report_result, table_by_table_quarterly_report_result)
# print(f"\n\n{len(diffs)} differences found between running the parser on the whole document and running it on each table")
# print(f"Diffs: {"\n".join(diffs)}")

# diffs = get_diffs(quarterly_report_result, all_tables_at_once_quarterly_report_result)
# print(f"\n\n{len(diffs)} differences found between running the parser on the whole document and running it on all of the tables at once")
# print(f"Diffs: {"\n".join(diffs)}")

# diffs = get_diffs(table_by_table_quarterly_report_result, all_tables_at_once_quarterly_report_result)
# print(f"\n\n{len(diffs)} differences found between running the parser on each table and running it on all of the tables at once")
# print(f"Diffs: {"\n".join(diffs)}")


## Calculations and Unusual Values

## Notes Taker

## Report Writer